In [3]:
# Basic Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


from sklearn.model_selection import train_test_split, RandomizedSearchCV

# Evaluation Metrics for Binary Classification
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score, 
    roc_auc_score, 
    confusion_matrix, 
    roc_curve
)

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
%pip install catboost lightgbm
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

  Using cached catboost-1.2.10-cp311-cp311-win_amd64.whl.metadata (1.5 kB)
  Using cached lightgbm-4.7.0-py3-none-win_amd64.whl.metadata (18 kB)
  Using cached graphviz-0.21-py3-none-any.whl.metadata (12 kB)
Using cached catboost-1.2.10-cp311-cp311-win_amd64.whl (100.2 MB)
Using cached lightgbm-4.7.0-py3-none-win_amd64.whl (1.4 MB)
Using cached graphviz-0.21-py3-none-any.whl (47 kB)
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.1/9.7 MB 656.4 kB/s eta 0:00:15
    --------------------------------------- 0.1/9.7 MB 901.1 kB/s eta 0:00:11
    --------------------------------------- 0.2/9.7 MB 1.2 MB/s eta 0:00:09
   - -------------------------------------- 0.3/9.7 MB 1.2 MB/s eta 0:00:08
   - -------------------------------------- 0.3/9.7 MB 1.2 MB/s eta 0:00:08
   - ------------------


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
# Load dataset
df = pd.read_csv('raw.csv')

# Drop identifier if present
if 'id' in df.columns:
    df = df.drop(columns=['id'])

# Show Top 5 Records
df.head()

,Gender,Age,Driving_License,Region_Code,Previously_Insured,Vehicle_Age,Vehicle_Damage,Annual_Premium,Policy_Sales_Channel,Vintage,Response
0,Male,44,1,28.0,0,> 2 Years,Yes,40454.0,26.0,217,1
1,Male,76,1,3.0,0,1-2 Year,No,33536.0,26.0,183,0
2,Male,47,1,28.0,0,> 2 Years,Yes,38294.0,26.0,27,1
3,Male,21,1,11.0,1,< 1 Year,No,28619.0,152.0,203,0
4,Female,29,1,41.0,1,< 1 Year,No,27496.0,152.0,39,0


In [8]:
# Preparing X and y variables (Target is 'Response')
X = df.drop(columns=['Response'])
y = df['Response']

# Preprocessing & Selection
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, RandomizedSearchCV
# Identify column types
num_features = X.select_dtypes(exclude="object").columns.tolist()
cat_features = X.select_dtypes(include="object").columns.tolist()

# Preprocessing Pipeline
numeric_transformer = StandardScaler()
oh_transformer = OneHotEncoder(drop='first', handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ("OneHotEncoder", oh_transformer, cat_features),
        ("StandardScaler", numeric_transformer, num_features)
    ]
)

# Fit-transform features
X = preprocessor.fit_transform(X)

# Split dataset into train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("X_train shape:", X_train.shape, "| X_test shape:", X_test.shape)

X_train shape: (304887, 11) | X_test shape: (76222, 11)


In [9]:
X

array([[ 1.        ,  0.        ,  1.        , ...,  0.57453868,
        -1.58723371,  0.74879538],
       [ 1.        ,  0.        ,  0.        , ...,  0.17263624,
        -1.58723371,  0.34244286],
       [ 1.        ,  0.        ,  1.        , ...,  0.4490531 ,
        -1.58723371, -1.52199808],
       ...,
       [ 1.        ,  1.        ,  0.        , ...,  0.26454281,
         0.88491205,  0.07950888],
       [ 0.        ,  0.        ,  1.        , ...,  0.81638891,
         0.22075349, -0.96027549],
       [ 1.        ,  0.        ,  0.        , ...,  0.6513986 ,
        -1.58723371,  0.98782627]], shape=(381109, 11))

In [10]:
def evaluate_model(true, predicted, predicted_proba):
    accuracy = accuracy_score(true, predicted)
    f1 = f1_score(true, predicted, average='binary', zero_division=0)
    precision = precision_score(true, predicted, average='binary', zero_division=0)
    recall = recall_score(true, predicted, average='binary', zero_division=0)
    roc_auc = roc_auc_score(true, predicted_proba)
    return accuracy, f1, precision, recall, roc_auc

In [11]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "XGBoost": XGBClassifier(eval_metric='logloss', random_state=42),
    "CatBoost": CatBoostClassifier(verbose=False, random_state=42),
    "LightGBM": LGBMClassifier(random_state=42)
}

model_list = []
roc_auc_list = []

for name, model in models.items():
    model.fit(X_train, y_train)
    
    # Predictions
    y_train_pred = model.predict(X_train)
    y_train_proba = model.predict_proba(X_train)[:, 1]
    
    y_test_pred = model.predict(X_test)
    y_test_proba = model.predict_proba(X_test)[:, 1]
    
    # Evaluate
    train_acc, train_f1, train_prec, train_rec, train_auc = evaluate_model(y_train, y_train_pred, y_train_proba)
    test_acc, test_f1, test_prec, test_rec, test_auc = evaluate_model(y_test, y_test_pred, y_test_proba)
    
    model_list.append(name)
    roc_auc_list.append(test_auc)
    
    print(name)
    print('Model performance for Training set')
    print("- Accuracy: {:.4f}".format(train_acc))
    print("- ROC-AUC: {:.4f}".format(train_auc))
    print("- F1 Score: {:.4f}".format(train_f1))
    print('----------------------------------')
    print('Model performance for Test set')
    print("- Accuracy: {:.4f}".format(test_acc))
    print("- ROC-AUC: {:.4f}".format(test_auc))
    print("- F1 Score: {:.4f}".format(test_f1))
    print('=' * 35)
    print('\n')

# Display Model Comparison Table
results_df = pd.DataFrame(list(zip(model_list, roc_auc_list)), columns=['Model Name', 'ROC_AUC_Score'])
results_df = results_df.sort_values(by="ROC_AUC_Score", ascending=False).reset_index(drop=True)
results_df

Logistic Regression
Model performance for Training set
- Accuracy: 0.8774
- ROC-AUC: 0.8363
- F1 Score: 0.0003
----------------------------------
Model performance for Test set
- Accuracy: 0.8774
- ROC-AUC: 0.8385
- F1 Score: 0.0002


Decision Tree
Model performance for Training set
- Accuracy: 0.9999
- ROC-AUC: 1.0000
- F1 Score: 0.9996
----------------------------------
Model performance for Test set
- Accuracy: 0.8233
- ROC-AUC: 0.5990
- F1 Score: 0.2950


Random Forest
Model performance for Training set
- Accuracy: 0.9999
- ROC-AUC: 1.0000
- F1 Score: 0.9995
----------------------------------
Model performance for Test set
- Accuracy: 0.8670
- ROC-AUC: 0.8334
- F1 Score: 0.1832


AdaBoost
Model performance for Training set
- Accuracy: 0.8774
- ROC-AUC: 0.8450
- F1 Score: 0.0000
----------------------------------
Model performance for Test set
- Accuracy: 0.8774
- ROC-AUC: 0.8458
- F1 Score: 0.0000


XGBoost
Model performance for Training set
- Accuracy: 0.8813
- ROC-AUC: 0.8803
- F

,Model Name,ROC_AUC_Score
0,LightGBM,0.856964
1,CatBoost,0.856613
2,XGBoost,0.855621
3,AdaBoost,0.845823
4,Logistic Regression,0.838516
5,Random Forest,0.833424
6,Decision Tree,0.599041


In [12]:
# Display Model Comparison Table
results_df = pd.DataFrame(list(zip(model_list, roc_auc_list)), columns=['Model Name', 'ROC_AUC_Score'])
results_df = results_df.sort_values(by="ROC_AUC_Score", ascending=False).reset_index(drop=True)
results_df

,Model Name,ROC_AUC_Score
0,LightGBM,0.856964
1,CatBoost,0.856613
2,XGBoost,0.855621
3,AdaBoost,0.845823
4,Logistic Regression,0.838516
5,Random Forest,0.833424
6,Decision Tree,0.599041


In [20]:
df['Vehicle_Damage']

0         Yes
1          No
2         Yes
3          No
4          No
         ... 
381104     No
381105     No
381106     No
381107    Yes
381108     No
Name: Vehicle_Damage, Length: 381109, dtype: str